In [21]:
import torch
import gc

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print("GPU memory cleared")
print(f"GPU memory available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

GPU memory cleared
GPU memory available: 15.83 GB
GPU memory allocated: 8.90 GB
GPU memory reserved: 15.28 GB


In [5]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
import shutil
from rank_bm25 import BM25Okapi


In [28]:
import os
import re
from typing import List, Optional
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from rank_bm25 import BM25Okapi


class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================
    def __init__(self, pdf_folder: str, embeddings: HuggingFaceEmbeddings, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.embeddings = embeddings

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.bm25_docs: List[Document] = []
        self.bm25_corpus: List[List[str]] = []

        # Initialize LLM for answer generation
        self.llm = self._init_llm()

    def _init_llm(self) -> Optional[HuggingFacePipeline]:
        """Initialize LLM for answer generation"""
        try:
            
            model_name = "Qwen/Qwen2.5-3B-Instruct"
            print("🤖 Loading LLM for answer generation..."+ model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32,
                device_map="auto",
                trust_remote_code=True
            )
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True,
                top_p=0.95
            )
            llm = HuggingFacePipeline(pipeline=pipe)
            print("✅ LLM loaded successfully")
            return llm
        except Exception as e:
            print(f"⚠️ Could not load Phi-2: {e}")
            print("Will use extractive answers only")
            return None

    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================
    def _detect_case_number(self, query: str) -> Optional[str]:
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None

    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Document]:
        if self.vectorstore is None or self.bm25 is None:
            print("⚠️ Vectorstore or BM25 not built yet.")
            return []

        # Semantic search
        semantic_results = self.vectorstore.similarity_search(query, k=k)

        # BM25 keyword search
        tokens = query.split()
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
        bm25_results = [self.bm25_docs[i] for i in bm25_top_idx]

        # Merge results (remove duplicates by case_number)
        combined = {}
        for doc in semantic_results + bm25_results:
            key = doc.metadata.get("case_number", doc.metadata.get("source_file", id(doc)))
            if key not in combined:
                combined[key] = doc

        return list(combined.values())[:k]

    # =======================================================================
    #                         SEARCH + ANSWER
    # =======================================================================
    def search(self, query: str, k: int = 10) -> str:
        """Search for documents and generate a structured legal answer."""
        print(f"\n🔍 QUERY: {query}")

        # Detect case number in query
        case_num = self._detect_case_number(query)

        if case_num:
            print(f"📋 Detected case number: {case_num}")
            parts = case_num.split("_")
            if len(parts) >= 2:
                case_number_only = parts[-2]
                case_year = parts[-1]

                # Retrieve more results to allow filtering
                results = self._retrieve_documents(query, k=k*3)

                # Filter to exact case match (number + year)
                exact_match = [
                    r for r in results
                    if case_number_only in r.metadata.get('case_number', '') and case_year == r.metadata.get('case_year', '')
                ]

                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match")
                else:
                    results = results[:k]
                    print(f"⚠️ No exact case found, returning top semantic matches")
            else:
                results = self._retrieve_documents(query, k=k)
        else:
            results = self._retrieve_documents(query, k=k)

        # Print only the filenames of retrieved PDFs
        print("\n📄 Retrieved PDFs:")
        for r in results:
            print(" -", r.metadata.get("source_file", "unknown"))

        # Generate answer using all retrieved documents
        answer_text = self.generate_answer(query, results)

        # Print clean structured answer
        print("\n📌 LLM Answer:\n")

        return answer_text

    # =======================================================================
    #                         ANSWER GENERATION
    # =======================================================================
    def generate_answer(self, query: str, documents: List[Document], use_llm: bool = True) -> str:
        """Generate a structured answer using all retrieved documents."""
        if not documents:
            return "No relevant documents found."

        # Build context from all documents
        context = "\n\n---\n\n".join([
            f"From {d.metadata.get('source_file', 'unknown')}:\n{d.page_content}"
            for d in documents
        ])

        if self.llm and use_llm:
            prompt = f"""

            You are a Legal Case Retrieval and Question Answering Assistant. You answer strictly using ONLY the retrieved documents provided to you.

You operate in two modes:

======================================================================
MODE 1 — STRICT CASE MODE
======================================================================
Triggered when the user query refers to a specific case number or appeal, e.g.:
"CPLA 210 of 2024", "Civil Appeal 152/2019", "C.P.L.A 47 2024", etc.

Rules:
1. You MUST identify the retrieved document(s) that correspond to that specific case, even if naming varies (e.g., CPLA / C.P.L.A / C.P.L.A. are treated as equivalent).
2. In STRICT mode, you may ONLY use the matching case document(s).
3. Ignore all other retrieved documents completely.
4. If the matching case is not present in the retrieved documents, respond:
   "The retrieved documents do not contain the required information."
5. You must never hallucinate missing details.

======================================================================
MODE 2 — LENIENT TOPIC MODE
======================================================================
Triggered when the user asks a general or semantic question, e.g.:
"cases on domestic violence", "similar cases", "cases about nomination papers", 
"precedents about election symbols", etc.

Rules:
1. You may use ANY of the retrieved documents.
2. You must synthesize everything into ONE unified answer.
3. Never produce multiple separate answers per document.
4. Quote ONLY text that appears in the retrieved documents.

======================================================================
UNIVERSAL RULES (APPLY TO BOTH MODES)
======================================================================
- Use only information contained in the retrieved documents.
- Never invent or guess facts (judges, parties, citations, reasoning, legal rules).
- If the answer is not found in the retrieved documents, say so explicitly.
- You must follow the output format EXACTLY as required below.
- Do NOT include analysis, chain of thought, system messages, reasoning steps, or metadata.
- Provide a single, clean, professional legal answer.

======================================================================
MANDATORY OUTPUT FORMAT
======================================================================

Answer:
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only include text you truly used.)

======================================================================

BEGIN NOW.
            
            """

            # Append retrieved documents to prompt
            prompt += "\n\nRETRIEVED DOCUMENTS:\n" + context

            try:
                result = self.llm.invoke(prompt)
                
                # Extract the generated text
                if isinstance(result, list) and len(result) > 0:
                    answer_text = result[0].get("generated_text", str(result))
                elif isinstance(result, str):
                    answer_text = result
                else:
                    answer_text = str(result)
                
                # Remove everything before "Answer:"
                if "Answer:" in answer_text:
                    answer_text = answer_text.split("Answer:")[-1].strip()
                
                # Remove common artifact patterns (but keep the actual answer)
                # Only remove if they appear at the START of the text
                start_artifacts = ["Q:", "Question:", "RETRIEVED", "From", "Documents:"]
                for artifact in start_artifacts:
                    if answer_text.startswith(artifact):
                        answer_text = answer_text.split("\n", 1)[-1].strip()
                
                # Clean up leading/trailing whitespace
                answer_text = answer_text.strip()
                
                print(answer_text)
                # Final output with sources
                return answer_text
                
            except Exception as e:
                print(f"⚠️ LLM error: {e}")
                return self._extractive_answer(documents)
        else:
            return self._extractive_answer(documents)


In [7]:
print("=" * 70)
print("COPYING DB TO WRITABLE LOCATION")
print("=" * 70)

# Copy from read-only input to writable workspace
src = "/kaggle/input/fyp-vector-store/chroma_db"
dst = "/kaggle/working/chroma_db"

if os.path.exists(dst):
    shutil.rmtree(dst)

print(f"\nCopying from: {src}")
print(f"Copying to: {dst}")
shutil.copytree(src, dst)
print("✅ Copied successfully")

# Now load from writable location
print("\n📚 Loading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'}
)

print("\n🔧 Initializing agent...")

COPYING DB TO WRITABLE LOCATION

Copying from: /kaggle/input/fyp-vector-store/chroma_db
Copying to: /kaggle/working/chroma_db
✅ Copied successfully

📚 Loading embeddings...


/tmp/ipykernel_48/529954725.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


🔧 Initializing agent...


In [29]:
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    embeddings=embeddings,
    db_path="/kaggle/working/chroma_db",  # ← Use writable location
    test_mode=False
)

print("\n📦 Loading vector store...")
agent.vectorstore = Chroma(
    persist_directory="/kaggle/working/chroma_db",
    embedding_function=embeddings
)
chunk_count = agent.vectorstore._collection.count()
print(f"✅ Loaded {chunk_count} chunks")

print("\n🔨 Rebuilding BM25...")
vectorstore_data = agent.vectorstore.get()

docs = []
for i, content in enumerate(vectorstore_data['documents']):
    metadata = vectorstore_data['metadatas'][i]
    doc = Document(page_content=content, metadata=metadata)
    docs.append(doc)

agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)
print(f"✅ BM25 ready with {len(agent.bm25_docs)} documents")

print("\n" + "=" * 70)
print("✅ READY TO USE")
print("=" * 70)

🤖 Loading LLM for answer generation...Qwen/Qwen2.5-3B-Instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


✅ LLM loaded successfully

📦 Loading vector store...
✅ Loaded 21895 chunks

🔨 Rebuilding BM25...
✅ BM25 ready with 21895 documents

✅ READY TO USE


In [30]:
query = "What was CPLA 210 of 2024 about?"
answer = agent.search(query, k=5)


🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
✅ Found exact case match

📄 Retrieved PDFs:
 - C.P.L.A.210_2024.pdf
[Write the single, synthesized answer here — do not include the prompt or retrieval metadata]

Sources used:
[List only the PDF filenames actually used. One per line, nothing else.]

Exact excerpts from sources:
"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

"[Copy exact text verbatim — word-for-word]"
— [PDF filename]

(Include as many excerpts as needed. Only include text you truly used.)


BEGIN NOW.
            
            

RETRIEVED DOCUMENTS:
From C.P.L.A.210_2024.pdf:
[Case Type: C.P.L.A.210] [Year: 2024] [Case Number: C.P.L.A.210_2024]

C.P.L.As.210 and 212/2024 
 
 
-:2:-
candidates for the General Elections of 2024. This candidate shall 
immediately and forthwith, and it shall be the duty of the Election 
Commission to ensure that this is done, be allocated an e lection 
symbol. (We may note that for this consti

In [27]:
 tests = [
    "CPLA-210/2024",
    "C.P.L.A 210 2024",
    "petition 210 of 2024 Supreme Court",
    "appeal 210 2024",
    "210 CPLA 2024",
    "Civil Appeal 2121 of 2017",
    "Leave to Appeal 210 2024"
]

for t in tests:
    print("\n=== TEST:", t, "===")
    docs = agent._retrieve_documents(t, k=20)
    for r in docs:
            print(" -", r.metadata.get("source_file", "unknown"))


=== TEST: CPLA-210/2024 ===
 - C.P.4_2021.pdf
 - C.P.L.A.210_2024.pdf
 - C.P.L.A.3644_2020.pdf
 - Crl.P.L.A.128_2024.pdf
 - Crl.P.L.A.80-P_2024.pdf
 - Crl.P.L.A.340_2024.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.3209_2020.pdf
 - C.A.316_2022.pdf
 - C.P.L.A.1354_2023.pdf
 - C.P.L.A.767_2022.pdf
 - C.R.P.1077_2023.pdf
 - C.P.L.A.1174_2022.pdf
 - C.P.L.A.2270_2019.pdf
 - C.A.1509_2021.pdf
 - C.P.L.A.1417_2022.pdf

=== TEST: C.P.L.A 210 2024 ===
 - Crl.P.L.A.80-P_2024.pdf
 - C.P.L.A.3644_2020.pdf
 - C.P.L.A.210_2024.pdf
 - C.P.L.A.184_2024.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.414_2021.pdf
 - Crl.P.L.A.340_2024.pdf
 - C.P.4_2021.pdf
 - Crl.P.L.A.128_2024.pdf
 - C.P.L.A.47_2024.pdf
 - Crl.P.L.A.458_2024.pdf
 - Crl.P.L.A.220_2024.pdf
 - C.P.L.A.2270_2019.pdf
 - C.R.P.870_2023.pdf
 - C.P.L.A.3531_2021.pdf
 - C.A.1349_2024.pdf
 - C.P.L.A.6-L_2023.pdf
 - C.P.L.A.1057_2019.pdf
 - C.P.L.A.2250-L_2016.pdf
 - C.A.2186_2017.pdf

=== TEST: petition 210 of 2024 Supreme Court ===
 - C.P.L.A.210_2024